# Chapitre 9 — DocuRAG

[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ahouahounko/rag-en-pratique/blob/main/chapters/chapitre-09-docurag/09_docurag.ipynb)

DocuRAG organise le pipeline du chapitre 2 en application modulaire. Le parcours reste entièrement hors ligne par défaut.

## 1. Préparer le dépôt

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

if not Path("src").is_dir():
    if not Path("rag-en-pratique").is_dir():
        subprocess.run(["git", "clone", "https://github.com/Ahouahounko/rag-en-pratique.git"], check=True)
    os.chdir("rag-en-pratique")

sys.path.insert(0, str(Path("src").resolve()))
print("Dépôt prêt :", Path.cwd())


## 2. Charger l'application DocuRAG

In [ ]:
from pathlib import Path
import sys

runnable = Path("chapters/chapitre-09-docurag/runnable").resolve()
sys.path.insert(0, str(runnable))

from docurag import DocuRAG
from docurag.config import Settings

settings = Settings(use_openai=False, chunk_size=60, chunk_overlap=10, top_k=3)
app = DocuRAG(settings)


## 3. Ingérer un dossier

In [ ]:
chunk_count = app.ingest(Path("data/sample"))
print(f"{chunk_count} chunks indexés")


## 4. Interroger DocuRAG

In [ ]:
result = app.ask("Quel est le délai de livraison standard ?")
print(result["answer"])


## 5. Inspecter la traçabilité

In [ ]:
for rank, source in enumerate(result["sources"], start=1):
    print(f"#{rank} score={source['score']} source={source['metadata']['source']}")
    print(source["text"][:300])
    print()


## 6. Tester une question absente des documents

In [ ]:
unknown = app.ask("Quel est le numéro de téléphone du directeur ?")
print(unknown["answer"])


## 7. Mode OpenAI facultatif

Cette cellule reste désactivée. Après configuration de votre clé, elle remplace les embeddings et la génération hors ligne par les API OpenAI.

In [ ]:
USE_OPENAI = False

if USE_OPENAI:
    online_settings = Settings.from_env()
    if not online_settings.use_openai:
        raise RuntimeError("Définissez DOCURAG_USE_OPENAI=true")
    online_app = DocuRAG(online_settings)
    online_app.ingest(Path("data/sample"))
    print(online_app.ask("Quel est le délai de livraison standard ?")["answer"])
else:
    print("Mode OpenAI désactivé : aucun appel API effectué.")


## Architecture

- `config.py` : configuration explicite ;
- `loaders.py` : chargement Markdown, texte et PDF facultatif ;
- `pipeline.py` : ingestion, retrieval et génération ;
- `cli.py` : interface en ligne de commande ;
- `rag_en_pratique.core` : composants partagés et testables.